In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from pyspark.sql import SparkSession

from pyspark.sql.functions import col, split, pandas_udf
from pyspark.ml.feature import Word2Vec
from typing import Iterator
from gensim.models import Word2Vec as GensimWord2Vec

In [2]:
!pip install gensim

In [3]:
import os
import sys
import re
# =========================================================
# 1. ÉP VERSION PYSPARK (PHẢI CHẠY ĐẦU TIÊN TRƯỚC KHI IMPORT)
# =========================================================
modules_to_remove = [mod for mod in sys.modules if mod.startswith('pyspark') or mod.startswith('py4j')]
for mod in modules_to_remove: 
    del sys.modules[mod]

sys.path = [p for p in sys.path if "/usr/local/spark" not in p]
if "PYTHONPATH" in os.environ: 
    del os.environ["PYTHONPATH"]
    
conda_site_packages = "/opt/conda/lib/python3.13/site-packages"
if conda_site_packages not in sys.path: 
    sys.path.insert(0, conda_site_packages)
    
os.environ["SPARK_HOME"] = os.path.join(conda_site_packages, "pyspark")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"


# =========================================================
# 2. BÂY GIỜ MỚI IMPORT PYSPARK VÀ KHỞI TẠO SESSION
# =========================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, udf
from pyspark.sql.types import FloatType
spark = SparkSession.builder \
    .appName("LSTM_word2vec_token_Pipeline") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://raw-financial-data/iceberg_warehouse") \
    .getOrCreate()


# =========================================================
# 3. VÁ LỖI THỜI GIAN HADOOP
# =========================================================
hadoop_conf = spark._jsc.hadoopConfiguration()
iterator = hadoop_conf.iterator()
while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower()
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num, unit = int(match.group(1)), match.group(2)
        ms_val = num * 1000 if unit == 's' else num * 60000 if unit == 'm' else num * 3600000 if unit == 'h' else num * 86400000
        hadoop_conf.set(entry.getKey(), str(ms_val))

print("✅ Khởi tạo Spark và môi trường hoàn tất!")

✅ Khởi tạo Spark và môi trường hoàn tất!


In [4]:
# Đọc bảng chứa dữ liệu, loại bỏ các dòng thiếu 4 cột điểm số
df_spark = spark.table("my_catalog.processed_zone_finnhub.news_tokens_sentiment_labeled").dropna(
    subset=["title_tokens", "summary_tokens", "title_polarity", "summary_polarity", "title_subjectivity", "summary_subjectivity"]
)

In [5]:
print("Đang tải dữ liệu về RAM (Pandas)...")
df_train = df_spark.toPandas()


Đang tải dữ liệu về RAM (Pandas)...


In [6]:
# Gộp title và summary thành một cột duy nhất để đưa vào Word2Vec và LSTM
df_train['tokenized_text'] = df_train['title_tokens'].astype(str) + " " + df_train['summary_tokens'].astype(str)

print(f"✅ Đã tải xong {len(df_train)} bài báo.")

✅ Đã tải xong 536055 bài báo.


In [7]:
# ==========================================
# 0. CẤU HÌNH THAM SỐ
# ==========================================
EMBEDDING_DIM = 100
MAX_SEQ_LEN = 50
HIDDEN_DIM = 64
EPOCHS = 10
BATCH_SIZE = 128


# ==========================================
# 2. HUẤN LUYỆN WORD2VEC BẰNG GENSIM (SIÊU TỐC)
# ==========================================
print("\n--- 2. Huấn luyện Word2Vec bằng Gensim ---")
# Tách các câu thành list các từ
sentences = [str(text).split() for text in df_train['tokenized_text']]

# Train Gensim Word2Vec (Dùng toàn bộ luồng CPU hiện có: workers=os.cpu_count())
w2v_model = GensimWord2Vec(sentences, vector_size=EMBEDDING_DIM, window=5, min_count=2, workers=os.cpu_count())
print("✅ Huấn luyện Word2Vec hoàn tất.")

# Tạo từ điển và Ma trận Embedding
vocab = w2v_model.wv.key_to_index
vocab_size = len(vocab) + 1 # +1 cho Padding
embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))

for word, idx in vocab.items():
    embedding_matrix[idx + 1] = w2v_model.wv[word]

embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)




--- 2. Huấn luyện Word2Vec bằng Gensim ---
✅ Huấn luyện Word2Vec hoàn tất.


In [8]:
# ==========================================
# 3. CHUẨN BỊ DỮ LIỆU CHO PYTORCH
# ==========================================
print("\n--- 3. Chuẩn bị DataLoader ---")
def text_to_indices(text, vocab_dict, max_len):
    tokens = str(text).split()
    indices = [vocab_dict.get(token, -1) + 1 for token in tokens] # +1 vì index 0 dành cho padding
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    return indices

class SentimentDataset(Dataset):
    def __init__(self, df, vocab_dict, max_len):
        self.texts = [text_to_indices(t, vocab_dict, max_len) for t in df['tokenized_text']]
        # Gom 4 cột điểm số thành ma trận nhãn (Labels) cho PyTorch
        self.scores = df[['title_polarity', 'summary_polarity', 'title_subjectivity', 'summary_subjectivity']].values
        
    def __len__(self):
        return len(self.scores)
        
    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long), torch.tensor(self.scores[idx], dtype=torch.float32)

dataset = SentimentDataset(df_train, vocab, MAX_SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)



--- 3. Chuẩn bị DataLoader ---


In [9]:
# ==========================================
# 4. KHỞI TẠO VÀ HUẤN LUYỆN BI-LSTM
# ==========================================

# Tự động nhận diện GPU (NVIDIA CUDA hoặc Apple Metal MPS) hoặc dùng CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("🚀 Đang sử dụng NVIDIA GPU để huấn luyện.")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Đang sử dụng Apple Silicon GPU để huấn luyện.")
else:
    device = torch.device("cpu")
    print("🐢 Đang sử dụng CPU để huấn luyện.")

class SentimentScoreLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, emb_matrix_tensor):
        super(SentimentScoreLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(emb_matrix_tensor)
        self.embedding.weight.requires_grad = False # Đóng băng Word2Vec để train nhanh hơn
        
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 4) # Đầu ra dự đoán 4 điểm cùng lúc
        
    def forward(self, text):
        embedded = self.embedding(text)
        _, (hidden, _) = self.lstm(embedded)
        hidden_concat = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return self.fc(hidden_concat)

print("\n--- 4. Bắt đầu huấn luyện LSTM ---")
model = SentimentScoreLSTM(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, embedding_tensor).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    for batch_texts, batch_scores in dataloader:
        # Đẩy dữ liệu lên GPU/CPU
        batch_texts, batch_scores = batch_texts.to(device), batch_scores.to(device)
        
        optimizer.zero_grad()
        preds = model(batch_texts) # Không dùng squeeze() vì output giờ là ma trận [batch, 4]
        loss = criterion(preds, batch_scores)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {epoch_loss/len(dataloader):.4f}")

# Lưu mô hình cứng vào máy để sau này dùng lại không cần train
torch.save(model.state_dict(), "financial_lstm_model.pth")
print("✅ Đã lưu mô hình LSTM tại: financial_lstm_model.pth")

🐢 Đang sử dụng CPU để huấn luyện.

--- 4. Bắt đầu huấn luyện LSTM ---
Epoch 1/10 | Loss: 0.0174
Epoch 2/10 | Loss: 0.0098
Epoch 3/10 | Loss: 0.0084
Epoch 4/10 | Loss: 0.0075
Epoch 5/10 | Loss: 0.0070
Epoch 6/10 | Loss: 0.0066
Epoch 7/10 | Loss: 0.0061
Epoch 8/10 | Loss: 0.0060
Epoch 9/10 | Loss: 0.0057
Epoch 10/10 | Loss: 0.0056
✅ Đã lưu mô hình LSTM tại: financial_lstm_model.pth


In [ ]:
# ==========================================
# 5. DỰ ĐOÁN VÀ LƯU KẾT QUẢ VÀO ICEBERG
# ==========================================
print("\n--- 5. Bắt đầu dự đoán trên toàn bộ dữ liệu ---")

# Chuyển mô hình sang chế độ đánh giá (Tắt Dropout, Batch Norm...)
model.eval()
all_predictions = []

# Tạo DataLoader mới nhưng đặt shuffle=False để giữ nguyên thứ tự dòng của df_train
infer_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

# Tiến hành dự đoán (Tắt tính toán Gradient để tiết kiệm RAM/VRAM)
with torch.no_grad():
    for batch_texts, _ in infer_loader:
        batch_texts = batch_texts.to(device)
        
        # Mô hình trả ra ma trận [batch_size, 4]
        preds = model(batch_texts).cpu().numpy() 
        all_predictions.extend(preds)

# Chuyển list kết quả thành Numpy Array để dễ tách cột
preds_array = np.array(all_predictions)

# Thêm 4 cột dự đoán vào Pandas DataFrame gốc
df_train['pred_title_polarity'] = preds_array[:, 0]
df_train['pred_summary_polarity'] = preds_array[:, 1]
df_train['pred_title_subjectivity'] = preds_array[:, 2]
df_train['pred_summary_subjectivity'] = preds_array[:, 3]

print("✅ Đã hoàn tất dự đoán và ghép cột.")

# Chuyển đổi Pandas DataFrame về lại Spark DataFrame
spark_df_result = spark.createDataFrame(df_train)

# Đặt tên bảng Iceberg mới để lưu kết quả (Bạn có thể đổi tên tùy ý)
output_table = "my_catalog.processed_zone_finnhub.news_lstm_predictions"

# Ghi kết quả thành một bảng mới trong Iceberg
print(f"⏳ Đang ghi dữ liệu vào Apache Iceberg tại bảng: {output_table} ...")
spark_df_result.write.format("iceberg").mode("overwrite").saveAsTable(output_table)

print("🎉 HOÀN TẤT! Dữ liệu kèm kết quả dự đoán đã nằm gọn trong hệ thống.")


--- 5. Bắt đầu dự đoán trên toàn bộ dữ liệu ---
✅ Đã hoàn tất dự đoán và ghép cột.
